# LeetCode 2357 — Make Array Zero by Subtracting Equal Amounts bằng Python

Link đề: https://leetcode.com/problems/make-array-zero-by-subtracting-equal-amounts/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting, Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu thao tác "trừ cùng một số `x` cho mọi phần tử dương" và vì sao **mỗi thao tác tối ưu xóa sạch đúng một giá trị khác nhau**.
2. Biết biến một bài mô phỏng thành bài **đếm số giá trị dương khác nhau (distinct)** bằng `set`.
3. Biết dùng `set`, `sorted`, `heapq`, mảng đánh dấu `seen[0..100]`, và `Counter`/`dict.keys()` để đếm giá trị khác nhau.
4. Viết 5 cách: mô phỏng đúng đề, `set` các số dương, sắp xếp rồi đếm chỗ đổi giá trị, mảng đánh dấu 101 ô, và `heap` lấy dần giá trị nhỏ nhất.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: đếm cả số `0`, đếm số phần tử dương thay vì số giá trị khác nhau, quên rằng chọn `x` nhỏ hơn mức tối đa không có lợi.

## 2. Đề bài & ví dụ

Cho mảng số nguyên không âm `nums`. Mỗi **thao tác**:

1. Chọn một số nguyên `x` thỏa `0 < x <= min(các phần tử dương của nums)` (x không vượt quá phần tử dương nhỏ nhất).
2. Trừ `x` khỏi **mọi phần tử dương** của `nums` (các phần tử bằng `0` giữ nguyên).

Trả về **số thao tác tối thiểu** để biến toàn bộ `nums` thành `0`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,5,0,3,5]` | `3` | chọn `x=1` -> `[0,4,0,2,4]`; chọn `x=2` -> `[0,2,0,0,2]`; chọn `x=2` -> `[0,0,0,0,0]` |
| 2 | `[0]` | `0` | đã toàn `0` |

Constraints: `1 <= nums.length <= 100`; `0 <= nums[i] <= 100`.

In [1]:
examples = [([1, 5, 0, 3, 5], 3), ([0], 0)]
for nums, expected in examples:
    a = list(nums)
    steps = 0
    log = [list(a)]
    while any(a):
        x = min(v for v in a if v > 0)       # chọn x là phần tử dương nhỏ nhất
        a = [v - x if v > 0 else 0 for v in a]
        steps += 1
        log.append(list(a))
    print(f"nums={nums}: các bước={log} -> {steps} thao tác (mong đợi {expected})")

nums=[1, 5, 0, 3, 5]: các bước=[[1, 5, 0, 3, 5], [0, 4, 0, 2, 4], [0, 2, 0, 0, 2], [0, 0, 0, 0, 0]] -> 3 thao tác (mong đợi 3)
nums=[0]: các bước=[[0]] -> 0 thao tác (mong đợi 0)


## 3. Kiến thức nền

### 3.1. Phần tử dương và `min` có điều kiện

"Phần tử dương" là phần tử `> 0`. Nhỏ nhất trong các phần tử dương: `min(v for v in nums if v > 0)`. Nếu không có phần tử dương thì `min` của dãy rỗng gây lỗi, nên dùng `default=...` hoặc kiểm tra trước.

In [2]:
nums = [1, 5, 0, 3, 5]
print(min(v for v in nums if v > 0))
print(min((v for v in nums if v > 0), default=None))
print(min((v for v in [0, 0] if v > 0), default=None))
try:
    min(v for v in [0, 0] if v > 0)
except ValueError as e:
    print("ValueError:", e)

1
1
None
ValueError: min() iterable argument is empty


### 3.2. `set` — loại phần tử trùng

`set(nums)` giữ mỗi giá trị một lần. `len(set(nums))` là số giá trị khác nhau. Có thể lọc bằng comprehension: `{v for v in nums if v > 0}` là tập các giá trị **dương** khác nhau.

In [3]:
nums = [1, 5, 0, 3, 5]
print(set(nums), len(set(nums)))
pos = {v for v in nums if v > 0}
print(pos, len(pos))

{0, 1, 3, 5} 4
{1, 3, 5} 3


### 3.3. Sắp xếp và đếm "chỗ đổi giá trị"

Sau `sorted(nums)`, các giá trị bằng nhau đứng liền nhau. Số giá trị khác nhau = 1 + số vị trí `i` mà `s[i] != s[i-1]` (hoặc đếm bằng `groupby`).

In [4]:
s = sorted([1, 5, 0, 3, 5])
print(s)
print(1 + sum(1 for i in range(1, len(s)) if s[i] != s[i - 1]))

[0, 1, 3, 5, 5]
4


### 3.4. Mảng đánh dấu (boolean array) theo giá trị

Giá trị chỉ `0..100`, nên mảng `seen = [False] * 101` đánh dấu giá trị đã gặp; đếm số ô `True` (trừ ô 0).

In [5]:
nums = [1, 5, 0, 3, 5]
seen = [False] * 101
for v in nums:
    seen[v] = True
print([i for i, b in enumerate(seen) if b], sum(seen[1:]))

[0, 1, 3, 5] 3


### 3.5. `heapq` — lấy dần phần tử nhỏ nhất

`heapq.heapify(list)` biến list thành min-heap; `heappop` trả phần tử nhỏ nhất, `O(log n)`. Dùng để mô phỏng "lấy giá trị dương nhỏ nhất rồi trừ".

In [6]:
import heapq

h = [5, 1, 3, 5]
heapq.heapify(h)
print(heapq.heappop(h), heapq.heappop(h), h)

1 3 [5, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thử `[1,5,0,3,5]` bằng tay, **chọn `x` lớn nhất có thể** (bằng phần tử dương nhỏ nhất):

- Dương nhỏ nhất `1`: trừ `1` -> `[0,4,0,2,4]` (giá trị `1` biến thành `0`).
- Dương nhỏ nhất `2`: trừ `2` -> `[0,2,0,0,2]` (giá trị `3` ban đầu, giờ là `2`, biến thành `0`).
- Dương nhỏ nhất `2`: trừ `2` -> `[0,0,0,0,0]` (giá trị `5` ban đầu).

Mỗi thao tác xóa sạch đúng **một giá trị ban đầu** (`1`, rồi `3`, rồi `5`). Các số `5` bằng nhau nên cùng bị xóa trong cùng một thao tác. Có **3** giá trị dương khác nhau: `1, 3, 5`. Đáp án `3`.

Trực giác: hai số bằng nhau luôn "đi cùng nhau" cho đến khi cùng về 0; hai số khác nhau **không thể** cùng về 0 trong cùng một thao tác (vì sau cùng một lần trừ, chênh lệch giữa chúng không đổi mà chỉ một số về 0 tại một thời điểm...). Nên đáp án = số giá trị dương **khác nhau**.

In [7]:
nums = [1, 5, 0, 3, 5]
distinct_positive = sorted({v for v in nums if v > 0})
print("các giá trị dương khác nhau:", distinct_positive, "-> đáp án", len(distinct_positive))
# mỗi thao tác trừ đúng hiệu giữa hai giá trị liên tiếp
prev = 0
for v in distinct_positive:
    print(f"trừ x = {v - prev} để giá trị {v} (ban đầu) về 0")
    prev = v

các giá trị dương khác nhau: [1, 3, 5] -> đáp án 3
trừ x = 1 để giá trị 1 (ban đầu) về 0
trừ x = 2 để giá trị 3 (ban đầu) về 0
trừ x = 2 để giá trị 5 (ban đầu) về 0


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng đề

**Ý tưởng:** lặp: chọn `x` = phần tử dương nhỏ nhất, trừ `x` cho mọi phần tử dương, đếm một thao tác; dừng khi không còn phần tử dương.

**Độ phức tạp:** mỗi thao tác `O(n)`; số thao tác tối đa là số giá trị dương khác nhau, `<= min(n, 100)`. Tổng `O(n * d)` với `d` là số giá trị khác nhau. **Nút thắt:** ta mô phỏng cả mảng dù chỉ cần biết có bao nhiêu giá trị khác nhau.

In [8]:
from typing import List

class SolutionSimulate:
    def minimumOperations(self, nums: List[int]) -> int:
        a = list(nums)
        steps = 0
        while any(v > 0 for v in a):
            x = min(v for v in a if v > 0)
            a = [v - x if v > 0 else 0 for v in a]
            steps += 1
        return steps

for arr in ([1, 5, 0, 3, 5], [0], [3, 3, 3], [2, 4]):
    print(arr, "->", SolutionSimulate().minimumOperations(arr))

[1, 5, 0, 3, 5] -> 3
[0] -> 0
[3, 3, 3] -> 1
[2, 4] -> 2


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** mỗi thao tác (chọn `x` = dương nhỏ nhất `m`) làm các phần tử bằng `m` thành `0`, còn các phần tử lớn hơn giảm đi `m` nhưng **thứ tự và sự khác nhau giữa các giá trị được giữ nguyên** (hiệu hai giá trị không đổi). Vì thế số giá trị dương khác nhau giảm **đúng 1** mỗi thao tác.

**Cận dưới:** mỗi thao tác chỉ đưa tối đa **một** giá trị khác nhau về `0` (hai giá trị khác nhau luôn cách nhau một khoảng không đổi, nên không thể cùng về 0 sau cùng một số lần trừ). Do đó cần ít nhất `d` thao tác, `d` là số giá trị dương khác nhau.

**Đạt cận:** chọn `x` = dương nhỏ nhất mỗi lần, đúng `d` thao tác.

**Quy tắc:** đáp án `= len({v for v in nums if v > 0})`.

## 7. Các cách tiếp theo

### Cách 2 — `set` các số dương

Một dòng: đếm số giá trị dương khác nhau.

In [9]:
from typing import List

class SolutionSet:
    def minimumOperations(self, nums: List[int]) -> int:
        return len({v for v in nums if v > 0})

for arr in ([1, 5, 0, 3, 5], [0], [3, 3, 3], [2, 4]):
    print(arr, "->", SolutionSet().minimumOperations(arr))

[1, 5, 0, 3, 5] -> 3
[0] -> 0
[3, 3, 3] -> 1
[2, 4] -> 2


**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(d)` (tập giá trị khác nhau).

### Cách 3 — Sắp xếp rồi đếm chỗ đổi giá trị

Sắp xếp tăng dần; duyệt, mỗi khi gặp giá trị dương **khác** giá trị liền trước thì cộng 1. Không cần `set`.

In [10]:
from typing import List

class SolutionSort:
    def minimumOperations(self, nums: List[int]) -> int:
        count, prev = 0, 0                      # prev=0: ban đầu coi như đã 'thấy' giá trị 0
        for v in sorted(nums):
            if v > prev:                        # giá trị dương mới, lớn hơn mọi giá trị trước
                count += 1
                prev = v
        return count

for arr in ([1, 5, 0, 3, 5], [0], [3, 3, 3], [2, 4]):
    print(arr, "->", SolutionSort().minimumOperations(arr))

[1, 5, 0, 3, 5] -> 3
[0] -> 0
[3, 3, 3] -> 1
[2, 4] -> 2


**Độ phức tạp Cách 3:** Time `O(n log n)`, Space `O(n)` cho bản đã sắp (hoặc `O(1)` nếu `nums.sort()` tại chỗ). Chú ý điều kiện `v > prev` thay cho `v != prev` để tự loại luôn số `0` (vì `prev` khởi đầu bằng `0`).

### Cách 4 — Mảng đánh dấu 101 ô

Giá trị chỉ từ `0` đến `100`: đánh dấu `seen[v] = True` rồi đếm số ô dương `True`. Không băm, hằng số nhỏ.

In [11]:
from typing import List

class SolutionMarks:
    def minimumOperations(self, nums: List[int]) -> int:
        seen = [False] * 101
        for v in nums:
            seen[v] = True
        return sum(seen[1:])

for arr in ([1, 5, 0, 3, 5], [0], [3, 3, 3], [2, 4]):
    print(arr, "->", SolutionMarks().minimumOperations(arr))

[1, 5, 0, 3, 5] -> 3
[0] -> 0
[3, 3, 3] -> 1
[2, 4] -> 2


**Độ phức tạp Cách 4:** Time `O(n + 101)`, Space `O(101)` = `O(1)`.

### Cách 5 — `heap` lấy dần giá trị nhỏ nhất (mô phỏng "thông minh")

Đẩy mọi giá trị dương vào min-heap. Lặp: lấy `m = heappop`, đếm một thao tác, rồi **bỏ mọi phần tử bằng `m`** (pop tiếp nếu đỉnh bằng `m`). Không cần trừ thật vì chỉ quan tâm số giá trị khác nhau.

In [12]:
import heapq
from typing import List

class SolutionHeap:
    def minimumOperations(self, nums: List[int]) -> int:
        heap = [v for v in nums if v > 0]
        heapq.heapify(heap)
        steps = 0
        while heap:
            m = heapq.heappop(heap)
            steps += 1
            while heap and heap[0] == m:        # các giá trị bằng m cùng về 0 trong thao tác này
                heapq.heappop(heap)
        return steps

for arr in ([1, 5, 0, 3, 5], [0], [3, 3, 3], [2, 4]):
    print(arr, "->", SolutionHeap().minimumOperations(arr))

[1, 5, 0, 3, 5] -> 3
[0] -> 0
[3, 3, 3] -> 1
[2, 4] -> 2


**Độ phức tạp Cách 5:** Time `O(n log n)`, Space `O(n)`. Dùng nhiều hơn mức cần thiết cho bài này, nhưng cho thấy cách "mô phỏng thông minh" khi quên mất công thức.

## 8. TRACE từng bước

Trace Cách 1 với `nums = [1,5,0,3,5]`:

| Thao tác | `nums` trước | `x` = dương nhỏ nhất | `nums` sau |
|---|---|---|---|
| 1 | `[1,5,0,3,5]` | 1 | `[0,4,0,2,4]` |
| 2 | `[0,4,0,2,4]` | 2 | `[0,2,0,0,2]` |
| 3 | `[0,2,0,0,2]` | 2 | `[0,0,0,0,0]` |

Số thao tác `3` = số giá trị dương khác nhau `{1,3,5}`.

In [13]:
def trace_simulate(nums):
    a = list(nums)
    step = 0
    print(f"nums={a}")
    while any(v > 0 for v in a):
        x = min(v for v in a if v > 0)
        b = [v - x if v > 0 else 0 for v in a]
        step += 1
        print(f"  thao tác {step}: x={x}: {a} -> {b}")
        a = b
    return step

print("kết quả:", trace_simulate([1, 5, 0, 3, 5]))
print("kết quả:", trace_simulate([4, 4, 9]))

nums=[1, 5, 0, 3, 5]
  thao tác 1: x=1: [1, 5, 0, 3, 5] -> [0, 4, 0, 2, 4]
  thao tác 2: x=2: [0, 4, 0, 2, 4] -> [0, 2, 0, 0, 2]
  thao tác 3: x=2: [0, 2, 0, 0, 2] -> [0, 0, 0, 0, 0]
kết quả: 3
nums=[4, 4, 9]
  thao tác 1: x=4: [4, 4, 9] -> [0, 0, 5]
  thao tác 2: x=5: [0, 0, 5] -> [0, 0, 0]
kết quả: 2


In [14]:
def trace_sort(nums):
    s = sorted(nums)
    prev, count = 0, 0
    print("đã sắp xếp:", s)
    for v in s:
        mới = v > prev
        if mới:
            count += 1
            prev = v
        print(f"  v={v}: {'giá trị dương mới' if mới else 'bỏ qua'} -> count={count}, prev={prev}")
    return count

print("kết quả:", trace_sort([1, 5, 0, 3, 5]))

đã sắp xếp: [0, 1, 3, 5, 5]
  v=0: bỏ qua -> count=0, prev=0
  v=1: giá trị dương mới -> count=1, prev=1
  v=3: giá trị dương mới -> count=2, prev=3
  v=5: giá trị dương mới -> count=3, prev=5
  v=5: bỏ qua -> count=3, prev=5
kết quả: 3


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        return len({v for v in nums if v > 0})

print(Solution().minimumOperations([1, 5, 0, 3, 5]))
print(Solution().minimumOperations([0]))

3
0


## 10. Vì sao thuật toán đúng

- **Cận dưới.** Xét hai giá trị dương khác nhau `u < v` trong `nums`. Sau mỗi thao tác cả hai giảm cùng một lượng (nếu chưa về `0`), nên `v - u` giữ nguyên tới khi `u` về `0`. Khi `u` về `0` thì `v` vẫn dương (còn `v - u > 0`), vì vậy `u` và `v` **không bao giờ cùng về 0 trong một thao tác**. Mỗi thao tác chỉ làm "chết" tối đa một giá trị khác nhau, nên cần `>= d` thao tác.
- **Đạt cận.** Mỗi lần chọn `x` bằng dương nhỏ nhất `m` thì mọi phần tử bằng `m` về `0`, còn mọi phần tử lớn hơn vẫn dương. Số giá trị dương khác nhau giảm đúng `1`. Sau `d` thao tác không còn dương.
- **Kết luận.** Đáp án chính là `d = số giá trị dương khác nhau` — các cách 2–5 đều tính đúng đại lượng đó.
- **Số `0` không tính.** `0` đã là mục tiêu; thao tác chỉ ảnh hưởng phần tử dương.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`<= 100`), `d` là số giá trị dương khác nhau (`<= min(n, 100)`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(n * d)` | `O(n)` |
| 2. `set` | `O(n)` | `O(d)` |
| 3. Sắp xếp rồi đếm | `O(n log n)` | `O(n)` |
| 4. Mảng đánh dấu 101 ô | `O(n)` | `O(1)` |
| 5. `heap` | `O(n log n)` | `O(n)` |

Với `n <= 100`, mọi cách đều tức thì. `set` một dòng là đẹp nhất để nộp.

## 12. Bộ test

Đối chiếu tất cả các cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed`.

In [16]:
import random

solutions = [SolutionSimulate(), SolutionSet(), SolutionSort(), SolutionMarks(), SolutionHeap(), Solution()]

fixed = [
    ([1, 5, 0, 3, 5], 3),
    ([0], 0),
    ([3, 3, 3], 1),                 # cùng giá trị: một thao tác
    ([2, 4], 2),
    ([0, 0, 0], 0),                 # toàn 0
    ([100], 1),                     # một số lớn nhất
    ([1, 2, 3, 4, 5], 5),           # mọi giá trị khác nhau
    ([5, 5, 0, 5, 0], 1),
    ([7, 1, 7, 1, 7], 2),
    (list(range(0, 101)), 100),     # từ 0 đến 100
]
for nums, expected in fixed:
    for s in solutions:
        assert s.minimumOperations(list(nums)) == expected, (type(s).__name__, nums[:6], expected)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 10 test cố định cho 6 cách


In [17]:
random.seed(2357)
for _ in range(1000):
    n = random.randint(1, 30)
    hi = random.choice([1, 3, 10, 100])
    nums = [random.randint(0, hi) for _ in range(n)]
    ref = SolutionSimulate().minimumOperations(list(nums))
    for s in solutions[1:]:
        assert s.minimumOperations(list(nums)) == ref, (type(s).__name__, nums)
print("1000 test ngẫu nhiên đối chiếu mô phỏng: OK")

1000 test ngẫu nhiên đối chiếu mô phỏng: OK


In [18]:
# chọn x nhỏ hơn mức tối đa không bao giờ tốt hơn: thử mọi dãy lựa chọn x với mảng nhỏ (vét cạn)
from functools import lru_cache

@lru_cache(maxsize=None)
def brute_all_choices(state):
    pos = [v for v in state if v > 0]
    if not pos:
        return 0
    m = min(pos)
    return 1 + min(brute_all_choices(tuple(v - x if v > 0 else 0 for v in state)) for x in range(1, m + 1))

random.seed(11)
for _ in range(200):
    nums = tuple(random.randint(0, 6) for _ in range(random.randint(1, 6)))
    assert brute_all_choices(nums) == Solution().minimumOperations(list(nums)), nums
print("vét cạn mọi cách chọn x: OK (đáp án = số giá trị dương khác nhau)")

vét cạn mọi cách chọn x: OK (đáp án = số giá trị dương khác nhau)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm cả số `0`

`len(set(nums))` bao gồm giá trị `0`, nên ví dụ 1 ra `4` thay vì `3`. `0` là mục tiêu, không cần thao tác.

In [19]:
def sai_dem_ca_0(nums):
    return len(set(nums))             # SAI: tính cả 0

for arr in ([1, 5, 0, 3, 5], [0]):
    print(arr, "sai:", sai_dem_ca_0(arr), "| đúng:", Solution().minimumOperations(arr))

[1, 5, 0, 3, 5] sai: 4 | đúng: 3
[0] sai: 1 | đúng: 0


### Lỗi 2: Đếm số phần tử dương thay vì số giá trị dương **khác nhau**

Hai số `5` cùng về `0` trong cùng một thao tác, nên chỉ tính một lần.

In [20]:
def sai_dem_phan_tu(nums):
    return sum(1 for v in nums if v > 0)       # SAI: đếm phần tử, không đếm giá trị khác nhau

for arr in ([1, 5, 0, 3, 5], [3, 3, 3]):
    print(arr, "sai:", sai_dem_phan_tu(arr), "| đúng:", Solution().minimumOperations(arr))

[1, 5, 0, 3, 5] sai: 4 | đúng: 3
[3, 3, 3] sai: 3 | đúng: 1


### Lỗi 3: Mô phỏng nhưng trừ `x` cho cả số `0` (sinh số âm)

Đề chỉ trừ khỏi phần tử **dương**. Trừ cả `0` tạo số âm, nên điều kiện dừng `any(v != 0 ...)` không bao giờ đúng và vòng lặp có thể chạy mãi.

In [21]:
def sai_tru_ca_0(nums):
    x = min(v for v in nums if v > 0)
    return [v - x for v in nums]                          # SAI: số 0 cũng bị trừ -> thành số âm

print("sai (sau 1 thao tác):", sai_tru_ca_0([1, 5, 0, 3, 5]))
print("đúng:", [v - 1 if v > 0 else 0 for v in [1, 5, 0, 3, 5]], "| đáp án bài:", Solution().minimumOperations([1, 5, 0, 3, 5]))

sai (sau 1 thao tác): [0, 4, -1, 2, 4]
đúng: [0, 4, 0, 2, 4] | đáp án bài: 3


### Lỗi 4: `min` của dãy rỗng khi mảng toàn `0`

`min(v for v in nums if v > 0)` khi không có số dương gây `ValueError`. Phải kiểm tra điều kiện lặp (`any(v > 0 ...)`) hoặc dùng `default`.

In [22]:
nums = [0, 0]
try:
    m = min(v for v in nums if v > 0)
except ValueError as e:
    print("ValueError:", e)
print("dùng default:", min((v for v in nums if v > 0), default=0))
print("đúng:", Solution().minimumOperations(nums))

ValueError: min() iterable argument is empty
dùng default: 0
đúng: 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy mô phỏng "mọi phần tử cùng bị trừ" -> chú ý **hiệu giữa các phần tử không đổi**; thường đáp án là số giá trị khác nhau.
- Đếm giá trị khác nhau -> `len(set(...))`; nhớ lọc điều kiện (ở đây chỉ số dương).
- Giá trị nhỏ (`<= 100`) -> mảng đánh dấu cũng được.
- Hỏi "số thao tác tối thiểu" -> tìm **cận dưới** và chỉ ra cách đạt cận.
- Khi không chắc, viết brute mô phỏng và so với công thức bằng test ngẫu nhiên (như Mục 12).

## 15. Biến thể của bài

- **Cho trước `x` cố định** (không được chọn): số thao tác để về 0 hoặc không thể.
- **Chỉ trừ `x` khỏi phần tử lớn nhất** hoặc phần tử nhỏ nhất (bài khác hẳn).
- **Mảng có số âm:** cần quy ước lại.
- **Trả về dãy `x` đã chọn** (chính là các hiệu liên tiếp của các giá trị dương khác nhau đã sắp).
- **Số thao tác tối thiểu khi mỗi thao tác chỉ trừ trên một đoạn liên tiếp** (bài Minimum Number of Operations to Make Array Zero kiểu khác, dùng tư duy chênh lệch).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | làm đúng đề | `O(n d)` | `O(n)` | hiểu đề, kiểm chứng |
| 2. `set` | đếm giá trị dương khác nhau | `O(n)` | `O(d)` | nộp bài, một dòng |
| 3. Sắp xếp | đếm chỗ đổi giá trị | `O(n log n)` | `O(n)` | không dùng `set` |
| 4. Mảng 101 ô | đánh dấu giá trị | `O(n)` | `O(1)` | giá trị nhỏ cố định |
| 5. `heap` | bỏ dần giá trị nhỏ nhất | `O(n log n)` | `O(n)` | mô phỏng thông minh |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().minimumOperations([2, 2, 0, 7, 7, 4])`.

**Bài 2 (tính tay).** Với `[3, 9, 3, 6]`, liệt kê các `x` đã chọn theo cách tối ưu và số thao tác.

**Bài 3 (điền TODO).** Hoàn thành `distinct_positive(nums)` trả về **danh sách đã sắp** các giá trị dương khác nhau (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `chosen_x(nums)` trả về dãy các `x` đã chọn trong cách tối ưu.

**Bài 5 (mini challenge).** Viết `steps_with_fixed_x(nums, x)`: số thao tác nếu **luôn** phải trừ đúng `x` (cố định) khỏi mọi phần tử dương (khi một phần tử dương `< x` thì về `0`); trả `-1` nếu không phù hợp theo quy ước của bạn (ví dụ yêu cầu mọi phần tử dương `>= x` mỗi lần).

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def distinct_positive_todo(nums):
    # return sorted({...})
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: các giá trị dương khác nhau là `2, 4, 7` -> 3. Bài 2: `{3, 6, 9}`.
- **Mức 2 (vừa):** Bài 3: `sorted({v for v in nums if v > 0})`. Bài 4: `x` là hiệu liên tiếp, có `x_0 = d_0`, `x_i = d_i - d_{i-1}`.
- **Mức 3 (sâu):** Bài 5: mô phỏng: `while any(v > 0)`; với quy ước đề bài của bạn, nếu có `0 < v < x` thì trả `-1`; mỗi lượt trừ `x` khỏi mọi phần tử dương.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1 và 2
assert Solution().minimumOperations([2, 2, 0, 7, 7, 4]) == 3
assert sorted({v for v in [3, 9, 3, 6] if v > 0}) == [3, 6, 9]
assert Solution().minimumOperations([3, 9, 3, 6]) == 3

# Bài 3
def distinct_positive(nums):
    return sorted({v for v in nums if v > 0})

# Bài 4
def chosen_x(nums):
    d = distinct_positive(nums)
    return [d[i] - (d[i - 1] if i else 0) for i in range(len(d))]

# Bài 5 (quy ước: mọi phần tử dương phải >= x ở mỗi lượt, nếu không trả -1)
def steps_with_fixed_x(nums, x):
    a = list(nums)
    steps = 0
    while any(v > 0 for v in a):
        if any(0 < v < x for v in a):
            return -1
        a = [v - x if v > 0 else 0 for v in a]
        steps += 1
    return steps

assert distinct_positive([2, 2, 0, 7, 7, 4]) == [2, 4, 7]
assert chosen_x([2, 2, 0, 7, 7, 4]) == [2, 2, 3]
assert chosen_x([1, 5, 0, 3, 5]) == [1, 2, 2]
assert chosen_x([0]) == []
assert steps_with_fixed_x([2, 4, 6], 2) == 3
assert steps_with_fixed_x([2, 5], 2) == -1
assert steps_with_fixed_x([0, 0], 3) == 0
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm giá trị dương khác nhau)

```python
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        return len({v for v in nums if v > 0})
```

**Checklist**

- [ ] Loại số `0` khỏi tập (chỉ đếm giá trị `> 0`)?
- [ ] Đếm số giá trị **khác nhau**, không phải số phần tử dương?
- [ ] Hiểu vì sao đáp án đạt cận dưới (mỗi thao tác xóa đúng một giá trị)?
- [ ] Nếu mô phỏng: chỉ trừ phần tử dương, và `min` có điều kiện không bị dãy rỗng?
- [ ] Đã thử `[0]`, `[3,3,3]`, `[1,2,3,4,5]`, `[1,5,0,3,5]`?
- [ ] Signature đúng `minimumOperations(self, nums: List[int]) -> int` và có `from typing import List`?